# Notebook 05: Advanced Models (Gradient Boosting)

Builds on `m3`, the geocoded feature set from `wk6_geocoding_baseline_m3.ipynb`, since that's the most reliable version of the data so far and it's already been benchmarked against Linear Regression, Decision Tree, and Random Forest. This notebook adds two gradient boosting models, XGBoost and LightGBM, and does light hyperparameter tuning as called for in the task prompt.

Same feature list, same train/val/test split, same evaluation function as `wk6`, so the comparison against the m3 baseline table isolates model choice as the only changed variable, the same isolation principle already used for the m1 vs m3 geocoding comparison.

## 1. Setup and Imports

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# pip install xgboost lightgbm --break-system-packages   (uncomment/run in a terminal if not already installed)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42

os.chdir(os.path.expanduser("~/Desktop/CAPropPredictor"))

## 2. Load m3 Data and Reproduce the Feature Set

Reusing the exact feature list, preprocessor, and evaluation function from `wk6_geocoding_baseline_m3.ipynb` rather than redefining them slightly differently, so nothing about the comparison changes except the model itself.

In [2]:
housingtrainm3 = pd.read_csv("CRMLSCleaned/housingtrainm3.csv")
housingvalm3 = pd.read_csv("CRMLSCleaned/housingvalm3.csv")
housingtestm3 = pd.read_csv("CRMLSCleaned/housingtestm3.csv")

numeric_feature_columns = [
    "Latitude", "Longitude",
    "ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "ParkingTotal", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "GarageSpaces",
    "LivingArea", "LotSizeSquareFeet", "AssociationFee", "YearBuilt", "Levels", "Stories",
]
categorical_feature_columns = ["City", "CountyOrParish", "MLSAreaMajor", "HighSchoolDistrict", "Flooring"]
feature_columns = numeric_feature_columns + categorical_feature_columns
non_feature_columns = ["ClosePrice", "SaleYearMonth"]

assert set(feature_columns) == set(housingtrainm3.columns) - set(non_feature_columns), (
    set(housingtrainm3.columns) - set(non_feature_columns) - set(feature_columns)
)

X_train, y_train = housingtrainm3[feature_columns], housingtrainm3["ClosePrice"]
X_val, y_val = housingvalm3[feature_columns], housingvalm3["ClosePrice"]
X_test, y_test = housingtestm3[feature_columns], housingtestm3["ClosePrice"]

def make_preprocessor(numeric_cols, categorical_cols):
    return ColumnTransformer(transformers=[
        ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
    ])

def evaluate(pipeline, X, y, label):
    preds = pipeline.predict(X)
    r2 = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)
    mape = mean_absolute_percentage_error(y, preds)
    mdape = np.median(np.abs((y - preds) / y))
    print(f"{label:>5s}: R2={r2:.4f}  MAE=${mae:,.0f}  MAPE={mape:.2%}  MdAPE={mdape:.2%}")
    return {"r2": r2, "mae": mae, "mape": mape, "mdape": mdape}

print(f"train: {X_train.shape}  val: {X_val.shape}  test: {X_test.shape}")

train: (128452, 23)  val: (11898, 23)  test: (11908, 23)


## 3. Why Gradient Boosting

Random Forest already got to R2=0.8773 on m3 by averaging many independent trees, which reduces variance but doesn't correct for the errors of any individual tree. Gradient boosting trees are built sequentially, each new tree is fit to the residual error of the ensemble so far, which tends to squeeze out bias that bagging alone can't reach. That's the reason to expect XGBoost or LightGBM to have a shot at beating 0.8773, not a guarantee, boosting is also more prone to overfitting a training set than bagging is, which is exactly why the tuning step below leans on the validation split rather than training R2 to make any decision.

## 4. Light Hyperparameter Tuning

The task prompt calls for light tuning, not an exhaustive search, so this is a small manual grid over the parameters that matter most for boosted trees: `n_estimators`, `max_depth` (or `num_leaves` for LightGBM), and `learning_rate`, evaluated on the validation split rather than cross-validation. Val R2 picks the config, the test set stays untouched until the final evaluation in Section 5, so the test metric stays an honest read on generalization.

In [3]:
xgb_param_grid = [
    {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.10},
    {"n_estimators": 500, "max_depth": 5, "learning_rate": 0.05},
    {"n_estimators": 800, "max_depth": 6, "learning_rate": 0.03},
]

lgbm_param_grid = [
    {"n_estimators": 300, "max_depth": -1, "num_leaves": 31, "learning_rate": 0.10},
    {"n_estimators": 500, "max_depth": -1, "num_leaves": 63, "learning_rate": 0.05},
    {"n_estimators": 800, "max_depth": -1, "num_leaves": 127, "learning_rate": 0.03},
]

def tune_on_val(model_class, param_grid, model_label):
    best_params, best_val_r2, best_pipeline = None, -np.inf, None
    for params in param_grid:
        pipeline = Pipeline([
            ("preprocess", make_preprocessor(numeric_feature_columns, categorical_feature_columns)),
            ("model", model_class(random_state=RANDOM_STATE, **params)),
        ])
        pipeline.fit(X_train, y_train)
        val_preds = pipeline.predict(X_val)
        val_r2 = r2_score(y_val, val_preds)
        print(f"{model_label} {params} -> val R2={val_r2:.4f}")
        if val_r2 > best_val_r2:
            best_params, best_val_r2, best_pipeline = params, val_r2, pipeline
    print(f"best {model_label} params: {best_params} (val R2={best_val_r2:.4f})")
    return best_params, best_pipeline

print("--- Tuning XGBoost on validation set ---")
xgb_best_params, xgb_pipeline = tune_on_val(XGBRegressor, xgb_param_grid, "XGBoost")

print("\n--- Tuning LightGBM on validation set ---")
lgbm_best_params, lgbm_pipeline = tune_on_val(LGBMRegressor, lgbm_param_grid, "LightGBM")

--- Tuning XGBoost on validation set ---


XGBoost {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1} -> val R2=0.8448


XGBoost {'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.05} -> val R2=0.8552


XGBoost {'n_estimators': 800, 'max_depth': 6, 'learning_rate': 0.03} -> val R2=0.8670
best XGBoost params: {'n_estimators': 800, 'max_depth': 6, 'learning_rate': 0.03} (val R2=0.8670)

--- Tuning LightGBM on validation set ---


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006521 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4651
[LightGBM] [Info] Number of data points in the train set: 128452, number of used features: 1544
[LightGBM] [Info] Start training from score 1189327.862688


LightGBM {'n_estimators': 300, 'max_depth': -1, 'num_leaves': 31, 'learning_rate': 0.1} -> val R2=0.8857


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010055 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4651
[LightGBM] [Info] Number of data points in the train set: 128452, number of used features: 1544
[LightGBM] [Info] Start training from score 1189327.862688


LightGBM {'n_estimators': 500, 'max_depth': -1, 'num_leaves': 63, 'learning_rate': 0.05} -> val R2=0.8929


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006259 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4651
[LightGBM] [Info] Number of data points in the train set: 128452, number of used features: 1544
[LightGBM] [Info] Start training from score 1189327.862688


LightGBM {'n_estimators': 800, 'max_depth': -1, 'num_leaves': 127, 'learning_rate': 0.03} -> val R2=0.8973
best LightGBM params: {'n_estimators': 800, 'max_depth': -1, 'num_leaves': 127, 'learning_rate': 0.03} (val R2=0.8973)


## 5. Final Test Evaluation

Best config from each model's validation sweep, evaluated once on the held-out test set.

In [4]:
results_advanced = {}

print("--- XGBoost (test) ---")
results_advanced["XGBoost"] = {"test": evaluate(xgb_pipeline, X_test, y_test, "test")}

print("\n--- LightGBM (test) ---")
results_advanced["LightGBM"] = {"test": evaluate(lgbm_pipeline, X_test, y_test, "test")}

--- XGBoost (test) ---
 test: R2=0.8634  MAE=$203,261  MAPE=16.27%  MdAPE=11.60%

--- LightGBM (test) ---


 test: R2=0.8933  MAE=$166,210  MAPE=12.46%  MdAPE=8.82%


## 6. Comparison Against the m3 Baseline

Linear Regression, Decision Tree, and Random Forest numbers are the already-established m3 (geocoded) results from `wk6_geocoding_baseline_m3.ipynb`, hardcoded here rather than rerun, so this table is the full model lineup on an identical feature set.

In [5]:
baseline_m3_results = {
    "LinearRegression": {"r2": 0.821782, "mae": 248655.754596, "mape": 0.226937, "mdape": 0.160824},
    "DecisionTree": {"r2": 0.774068, "mae": 231738.962719, "mape": 0.169448, "mdape": 0.112363},
    "RandomForest": {"r2": 0.877303, "mae": 169443.678201, "mape": 0.122315, "mdape": 0.078733},
}

comparison_rows = []
for model_name, metrics in baseline_m3_results.items():
    comparison_rows.append({"model": model_name, **metrics})
for model_name, res in results_advanced.items():
    comparison_rows.append({"model": model_name, **res["test"]})

comparison_df = pd.DataFrame(comparison_rows).set_index("model").sort_values("r2", ascending=False)
comparison_df

,r2,mae,mape,mdape
model,,,,
LightGBM,0.893299,166210.376255,0.124633,0.088176
RandomForest,0.877303,169443.678201,0.122315,0.078733
XGBoost,0.863357,203261.331841,0.162707,0.115994
LinearRegression,0.821782,248655.754596,0.226937,0.160824
DecisionTree,0.774068,231738.962719,0.169448,0.112363


## 7. Persist the Best Models

Saving both fitted pipelines to disk rather than retraining boosted models inside the Week 8 evaluation notebook, they're the slowest models in the lineup and there's no reason to pay that cost twice. Linear Regression, Decision Tree, and Random Forest are cheap enough to just retrain directly in Notebook 06.

In [6]:
os.makedirs("models", exist_ok=True)
joblib.dump(xgb_pipeline, "models/xgb_m3.pkl")
joblib.dump(lgbm_pipeline, "models/lgbm_m3.pkl")
print("saved models/xgb_m3.pkl and models/lgbm_m3.pkl")

saved models/xgb_m3.pkl and models/lgbm_m3.pkl


## Reflection

Going in, the expectation was that gradient boosting would beat Random Forest, since boosting corrects the ensemble's residual error tree by tree instead of just averaging independent trees, and Random Forest was already the strongest of the three m3 baseline models at R2=0.8773. That held for one of the two models and not the other, which is itself the more interesting result than either number alone.

**LightGBM won, and by a margin that looks like signal, not noise.** LightGBM reached test R2=0.8933, MAE=6,210, MAPE=12.46%, MdAPE=8.82%, a 0.016 R2 improvement over Random Forest's 0.8773. That is an order of magnitude larger than the swings seen in the m1 vs m3 geocoding comparison (differences of 0.0001-0.008 that turned out to be noise), and MAE dropped by about \,200 on top of Random Forest already being the best of the first three models. Val R2 for the winning LightGBM configuration () was 0.8973, close to its test R2, so there's no sign the validation-set tuning step overfit to that split.

**XGBoost did not beat Random Forest.** XGBoost's best validation configuration (, the same point on the grid that won for LightGBM) reached test R2=0.8634, MAE=,261, below Random Forest's 0.8773. Both boosters were given the same three-point grid and the same validation-based selection rule, so the gap between them isn't a tuning artifact on one side, LightGBM's leaf-wise growth with 127 leaves is finding structure that XGBoost's depth-6, ~64-leaf-per-tree limit at the same learning rate and tree count doesn't reach in this feature set. A deeper or wider XGBoost grid might close some of that gap, but that's future tuning work, not a claim this notebook can make on the grid actually run.

**Overall model ranking on m3 (test R2): LightGBM (0.8933) > Random Forest (0.8773) > XGBoost (0.8634) > Linear Regression (0.8218) > Decision Tree (0.7741).** The Decision Tree's weak test score alongside its near-perfect training score (documented in notebook 4) is still the clearest overfitting signature in the lineup; XGBoost's and Random Forest's scores landing close together despite very different algorithms is a second useful data point that the ceiling for this feature set, whatever it is, sits somewhere in the high 0.8s for tree-based models, with LightGBM the only one so far to meaningfully clear it.

Worth carrying into Week 8: LightGBM's overall MAPE (12.46%) is close to Random Forest's (12.23%) even though its R2 is clearly better and its MAE and MdAPE are both lower, which suggests LightGBM's advantage isn't uniform across the price distribution, at minimum worth checking the price-band breakdown for whether LightGBM's gain is concentrated in one tier or spread evenly, the same question notebook 6 asks of every model in the lineup.
